In [1]:
# Student Grade Prediction


In [2]:
import os
import json
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


import joblib

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [3]:
rng = np.random.default_rng(42)

N = 2000

attendence = rng.uniform(40,100,N)

study_hours = rng.uniform(0,8,N)

previous_score = rng.uniform(20,100,N)

assignment_completed = rng.integers(0,11,N)

noise = rng.normal(0,10,N)

preformance_score = (
    0.25*attendence +
    4*study_hours +
    0.35*previous_score +
    2*assignment_completed +
    noise
)

passed = (preformance_score >= 75).astype(int)

df = pd.DataFrame({
    "attendence": attendence,
    "study_hours": study_hours,
    "previous_score": previous_score,
    "assignment_completed": assignment_completed,
    "passed": passed
})

df.head()

,attendence,study_hours,previous_score,assignment_completed,passed
0,86.437363,6.741856,77.287998,5,0
1,66.332706,3.571116,48.569248,5,0
2,91.515875,7.623801,88.149428,0,0
3,81.842082,5.206357,39.278173,1,0
4,45.650641,0.927191,63.071479,6,0


In [4]:
# EDA
# Shape
# IsNULL
# How many Passed
# Summary Stats
# Visualize each column

In [5]:
# Train Test Split

features = [
    "attendence",
    "study_hours",
    "previous_score",
    "assignment_completed"
]

target = "passed"

X = df[features]

y = df[target]

X_train,X_test,y_train,y_test= train_test_split(
    X,y,
    test_size=0.2,
    random_state=182
)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(1600, 4)
(1600,)
(400, 4)
(400,)


In [6]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression())
    ]),
    "Decision Tree" : DecisionTreeClassifier(max_depth = 4),
    "Random Forest" : RandomForestClassifier(n_estimators = 100,max_depth = 5),
    "SVM" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),
    "KNN" : KNeighborsClassifier(n_neighbors =3)
}

results = []

trained_models = {}

for model_name,model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    results.append({
        "model": model_name,
        "accuracy": accuracy,
        "confusion_matrix": confusion_matrix(y_test,y_pred).tolist(),
        "classification_report": classification_report(y_test,y_pred,output_dict=True)
    })
    trained_models[model_name] = model

results_df = pd.DataFrame(results)

results_df.head()

,model,accuracy,confusion_matrix,classification_report
0,Logistic Regression,0.8425,"[[275, 19], [44, 62]]","{'0': {'precision': 0.8620689655172413, 'recal..."
1,Decision Tree,0.8025,"[[264, 30], [49, 57]]","{'0': {'precision': 0.8434504792332268, 'recal..."
2,Random Forest,0.8375,"[[278, 16], [49, 57]]","{'0': {'precision': 0.8501529051987767, 'recal..."
3,SVM,0.8400,"[[279, 15], [49, 57]]","{'0': {'precision': 0.850609756097561, 'recall..."
4,KNN,0.7975,"[[264, 30], [51, 55]]","{'0': {'precision': 0.8380952380952381, 'recal..."
